In [1]:
# [Cell 0] Install Dependencies
!pip install -q \
    "opentelemetry-api>=1.39.0,<=1.42.1" \
    "opentelemetry-sdk>=1.39.0,<=1.42.1" \
    transformers \
    accelerate \
    chromadb \
    sentence-transformers

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 2.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 60.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 23.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.6/4.6 MB 87.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.6/23.6 MB 51.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 5.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 95.7/95.7 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 2.5 MB/s eta 0:00:00


In [2]:
# [Cell 1] Load Base Model and Tokenizer
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

model_name = "Qwen/Qwen2.5-1.5B-Instruct"

# 1. Load the tokenizer and ensure padding tokens are mapped
tokenizer = AutoTokenizer.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token

# 2. Load the base model in FP16 precision directly onto the GPU
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="cuda",
    torch_dtype=torch.float16
)

# Enable KV caching for fast inference generation
model.config.use_cache = True

print("Base model loaded successfully into GPU memory.")

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Base model loaded successfully into GPU memory.


In [3]:
# [Cell 2] Baseline Test Without Context
test_query = "Who leads the neurology department at MediCore Hospital?"

messages = [
    {"role": "user", "content": test_query}
]

# Format using Qwen's ChatML template
prompt_text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(prompt_text, return_tensors="pt").to(model.device)

# Generate response using greedy decoding
output_ids = model.generate(
    **inputs,
    max_new_tokens=60,
    do_sample=False,
    eos_token_id=tokenizer.eos_token_id
)

# Strip out the input prompt and decode newly generated tokens
response = tokenizer.decode(output_ids[0][inputs.input_ids.shape[1]:], skip_special_tokens=True)

print("BASELINE UNGROUNDED ANSWER:\n", response)

BASELINE UNGROUNDED ANSWER:
 I'm sorry, but I can't answer this question. This might be a sensitive and personal matter that should be discussed with a healthcare professional or contacted directly through their official channels. As an AI language model, I don't have access to private information about individuals or organizations. If you have any other


In [4]:
# [Cell 3] Ingest MediCore Knowledge into ChromaDB
import json
import chromadb
from chromadb.utils import embedding_functions

# 1. Download dataset if missing
!wget -nc -q https://raw.githubusercontent.com/AI-Learning-Repo/Data-Handling/refs/heads/week4/datasets/MediCore.json

# 2. Initialize in-memory Chroma client
chroma_client = chromadb.Client()
st_fn = embedding_functions.SentenceTransformerEmbeddingFunction(model_name="all-MiniLM-L6-v2")

collection = chroma_client.get_or_create_collection(
    name="medicore_rag",
    embedding_function=st_fn
)

# 3. Read and index records
with open("MediCore.json", "r", encoding="utf-8") as f:
    lines = [json.loads(line.strip()) for line in f]

collection.add(
    documents=[item["completion"] for item in lines],
    ids=[f"fact_{idx}" for idx in range(len(lines))],
    metadatas=[{"prompt": item["prompt"]} for item in lines]
)

print(f"ChromaDB ready: {collection.count()} knowledge chunks indexed.")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

ChromaDB ready: 486 knowledge chunks indexed.


In [5]:
# [Cell 4] Context Retrieval and Grounded Generation
user_query = "Who leads the neurology department at MediCore Hospital?"

# 1. Retrieve the top 2 most relevant passages
search_results = collection.query(
    query_texts=[user_query],
    n_results=2
)
retrieved_chunks = search_results["documents"][0]
context_block = "\n".join([f"- {chunk}" for chunk in retrieved_chunks])

print("--- RETRIEVED CONTEXT ---")
print(context_block)
print("--------------------------\n")

# 2. Build the Grounded ChatML Message Structure
messages = [
    {
        "role": "system",
        "content": (
            "You are a helpful assistant for MediCore Hospital. "
            "Answer the user's question relying ONLY on the provided context below. "
            "If the answer cannot be determined from the context, state: 'Information not available.'"
        )
    },
    {
        "role": "user",
        "content": f"Context:\n{context_block}\n\nQuestion: {user_query}"
    }
]

# 3. Apply Chat Template
formatted_prompt = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

# 4. Generate Answer
inputs = tokenizer(formatted_prompt, return_tensors="pt").to(model.device)

with torch.no_grad():
    output_tokens = model.generate(
        **inputs,
        max_new_tokens=80,
        do_sample=False,
        eos_token_id=tokenizer.eos_token_id
    )

grounded_response = tokenizer.decode(
    output_tokens[0][inputs.input_ids.shape[1]:],
    skip_special_tokens=True
).strip()

print("GROUNDED RAG ANSWER:\n", grounded_response)

--- RETRIEVED CONTEXT ---
- The neurology department handles brain and nervous system diseases at MediCore Hospital.
- MediCore Hospital includes emergency care, cardiology, neurology, oncology, pediatrics, radiology, orthopedics, psychiatry, internal medicine, and robotic surgery departments.
--------------------------

GROUNDED RAG ANSWER:
 The neurology department at MediCore Hospital is led by specialists who handle brain and nervous system diseases. However, without specific information about individual leaders within this department, it's not possible to determine who exactly leads them based solely on the given context. Therefore, the appropriate response is:

Information not available.


In [6]:
# [Cell 5] Negative Constraint Evaluation
unrecorded_query = "What is the name of MediCore Hospital's chief veterinary surgeon?"

# 1. Retrieve context for a non-existent topic
results = collection.query(query_texts=[unrecorded_query], n_results=2)
context_block = "\n".join([f"- {c}" for c in results["documents"][0]])

# 2. Format grounded prompt
messages = [
    {
        "role": "system",
        "content": (
            "You are a medical assistant for MediCore Hospital. "
            "Answer the question using ONLY the provided context. "
            "If the answer is not explicitly mentioned in the context, reply: 'Information not available.'"
        )
    },
    {
        "role": "user",
        "content": f"Context:\n{context_block}\n\nQuestion: {unrecorded_query}"
    }
]

prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

with torch.no_grad():
    output = model.generate(**inputs, max_new_tokens=40, do_sample=False)

reply = tokenizer.decode(output[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()

print(f"QUERY: {unrecorded_query}")
print(f"MODEL REFUSAL: {reply}")

QUERY: What is the name of MediCore Hospital's chief veterinary surgeon?
MODEL REFUSAL: Information not available.


In [7]:
# [Cell 6] Dynamic Knowledge Update Without Training
update_query = "Who leads the neurology department at MediCore Hospital?"

# 1. Update the document in ChromaDB
# Scenario: Dr. Elena Varga retired; Dr. Arto Virtanen was appointed
collection.update(
    ids=["fact_56"],
    documents=["Dr. Arto Virtanen leads the neurology department at MediCore Hospital."]
)

# 2. Re-run retrieval with the exact same query
results = collection.query(query_texts=[update_query], n_results=1)
updated_context = results["documents"][0][0]

messages = [
    {
        "role": "system",
        "content": "Answer the question strictly based only on the provided context."
    },
    {"role": "user", "content": f"Context:\n{updated_context}\n\nQuestion: {update_query}"}
]

prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

with torch.no_grad():
    output = model.generate(**inputs, max_new_tokens=50, do_sample=False)

updated_response = tokenizer.decode(output[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()

print("UPDATED RAG RESPONSE:\n", updated_response)

UPDATED RAG RESPONSE:
 Based solely on the given context, there is no explicit information about who leads the neurology department at MediCore Hospital. The context states that the neurology department deals with brain and nervous system diseases but does not mention any specific leadership or management details for


In [8]:
# [Appendix Cell] Python-Enforced JSON Output with Source Citations
import json

def generate_rag_json(query):
    # 1. Retrieve
    retrieved = collection.query(query_texts=[query], n_results=2)
    chunks = retrieved["documents"][0]
    sources = retrieved["ids"][0]

    # 2. Generate
    context_str = "\n".join([f"[{sid}] {c}" for sid, c in zip(sources, chunks)])
    messages = [
        {"role": "system", "content": "Answer the question using the context. Keep it under 20 words."},
        {"role": "user", "content": f"Context:\n{context_str}\n\nQuestion: {query}"}
    ]

    prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=50, do_sample=False)

    answer_text = tokenizer.decode(out[0][inputs.input_ids.shape[1]:], skip_special_tokens=True).strip()

    # 3. Python-enforced structured envelope
    payload = {
        "query": query,
        "answer": answer_text,
        "sources_used": sources
    }

    return json.dumps(payload, indent=2)

print(generate_rag_json("What is the address of MediCore Hospital?"))

{
  "query": "What is the address of MediCore Hospital?",
  "answer": "The address of MediCore Hospital is www.medicorehospital.fi.",
  "sources_used": [
    "fact_19",
    "fact_20"
  ]
}
